# Displacement autocorrelation functions

The displacement autocorrelation function $C(t_1, t_2)$ is computed for
each signal following the framework of Vollmer et al. (2021). It captures
how the displacement process at time $t_1$ is correlated with its value
at time $t_2$:

$$C(t_1, t_2) = \langle x(t_1)\, x(t_2) \rangle$$

The function is evaluated on a logarithmic grid of $(t_1, t_2)$ pairs to
capture scaling behavior across multiple time scales. The scaling exponent
$\beta$ is estimated from the diagonal $C(t, t) \sim t^\beta$.

Three definitions of the process $x(t)$ are explored, consistent with
the choices made in Section 3.

## 1. Imports and visualization settings

In [ ]:
from pathlib import Path
import pandas as pd
import logging
from src.signals_autocorrelation import (compute_displacement_autocorrelation, analyze_autocorrelation_scaling)
from src.plot_settings import set_plot_style
colors = set_plot_style()
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger()
def check(condition, message):
    if condition:
        logger.info(message)
    else:
        raise ValueError(message)
logger.info("Environment ready")
logger.info("Libraries imported successfully")

## 2. Data loading

Two preprocessed datasets are loaded:
- `acc_preprocessed_all.parquet`: all 66 signals, used only for the autocorrelation analysis.
- `acc_preprocessed_long.parquet`: 48 signals with at least 48,000 samples, used for the moment scaling analysis (short-signal stations excluded).

In [ ]:
df_acc_clean = pd.read_parquet('../data/processed/acc_preprocessed_all.parquet')
df_acc_long  = pd.read_parquet('../data/processed/acc_preprocessed_long.parquet')
print("All signals:", df_acc_clean['file'].nunique())
print("Long signals:", df_acc_long['file'].nunique())
# Figures output directory
FIGURES_DIR = Path('../figures/03_single_signal')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
(FIGURES_DIR / 'scaling' / 'displacement' / 'event_window').mkdir(parents=True, exist_ok=True)

## 3. Acceleration

The process is defined as $x(t) = a(t)$, the normalized acceleration signal.

In [ ]:
df_autocorr_acc, C_matrices_acc = compute_displacement_autocorrelation(
    df_acc_long,
    output_dir=FIGURES_DIR / 'autocorrelation' / 'acceleration'
)

### Scaling behavior

The scaling exponent $\beta$ is estimated from the diagonal $C(t, t)$,
which is expected to scale as:

$$C(t, t) = \langle x(t)^2 \rangle \sim t^\beta$$

The exponent $\beta$ is estimated by linear regression of
$\log C(t, t)$ vs $\log t$. A value of $\beta = 1$ corresponds to
normal diffusion ($\eta = 1$), while $\beta \neq 1$ indicates anomalous
scaling. In the framework of Vollmer et al. (2021), $\beta$ is directly
related to the mean-square displacement exponent $\eta$.

In [ ]:
df_autocorr_scaling_acc = analyze_autocorrelation_scaling(
    C_matrices_acc,
    output_dir=FIGURES_DIR / 'autocorrelation' / 'acceleration')
try:
    df_autocorr_scaling_acc.to_parquet(
        '../data/processed/autocorr_scaling_acc.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

## Velocity

The process is defined as the ground velocity:
$$v(t) = \sum_{k=0}^{t} a(k)\,\Delta t$$

In [ ]:
df_autocorr_vel, C_matrices_vel = compute_displacement_autocorrelation(
    df_acc_long,
    output_dir=FIGURES_DIR / 'autocorrelation' / 'velocity'
)
df_autocorr_scaling_vel = analyze_autocorrelation_scaling(
    C_matrices_vel,
    output_dir=FIGURES_DIR / 'autocorrelation' / 'velocity'
)
try:
    df_autocorr_scaling_vel.to_parquet(
        '../data/processed/autocorr_scaling_vel.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")


## Displacement

The process is defined as the ground displacement:
$$v(t) = \sum_{k=0}^{t} a(k)\,\Delta t, \qquad x(t) = \sum_{k=0}^{t} v(k)\,\Delta t$$

In [ ]:
df_autocorr_disp, C_matrices_disp = compute_displacement_autocorrelation(
    df_acc_long,
    output_dir=FIGURES_DIR / 'autocorrelation' / 'displacement'
)
df_autocorr_scaling_disp = analyze_autocorrelation_scaling(
    C_matrices_disp,
    output_dir=FIGURES_DIR / 'autocorrelation' / 'displacement'
)
try:
    df_autocorr_scaling_disp.to_parquet(
        '../data/processed/autocorr_scaling_disp.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

## Summary